# Example 4.4.2 / 2: The First Period Has No Predecessor

Companion notebook for **4.4.2 Multiperiod Models** in the AMPL lab Documentation, section "The First Period Has No Predecessor". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## The First Period Has No Predecessor

> **Pitfall: `prev` of the first period**
> Indexing `Balance` over every week makes AMPL compute `prev` of the first week, which
> doesn't exist. The model reads without complaint, and the error comes when the constraint is
> generated, at the solve. From Python, `ampl.solve()` logs the error, and
> `ampl.solve_result` is `?`: nothing was solved.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                     # products
set WEEKS ordered;            # weeks, in order
param inv0 {PROD} >= 0;       # inventory before the first week
var Make {PROD, WEEKS} >= 0;  # tons produced
var Inv {PROD, WEEKS} >= 0;   # tons in inventory at the end of the week
var Sell {PROD, WEEKS} >= 0;  # tons sold

subject to Balance {p in PROD, t in WEEKS}:  # prev(t) has no value for the first week
    Make[p,t] + Inv[p,prev(t)] = Sell[p,t] + Inv[p,t];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands"]
ampl.set["WEEKS"] = ["27sep", "04oct"]
ampl.param["inv0"] = {"bands": 10}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)

[ERROR] 
	Error executing "solve" command:
	error processing constraint Balance['bands','27sep']:
		can't compute prev('27sep', WEEKS) -- '27sep' is the first member


?


### Pure AMPL: data.dat and model.run

Pure AMPL, where `solve` prints the message:

In [6]:
# amplpy raises the previous solve's warning at the next statement; clear it before this run
try:
    ampl.eval("reset;")
except Exception:
    pass

In [7]:
%%writefile data.dat
set PROD := bands;
set WEEKS := 27sep 04oct;
param inv0 := bands 10;

Writing data.dat


In [8]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [9]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
option solver highs;
solve;
""")
except Exception:
    pass  # AMPL has printed the error above

[ERROR] 
	Error executing "solve" command:
	error processing constraint Balance['bands','27sep']:
		can't compute prev('27sep', WEEKS) -- '27sep' is the first member
